In [1]:
import numpy as np
import pandas as pd
from huggingface_hub import login, HfApi, hf_hub_download
from sklearn.metrics.pairwise import cosine_similarity
from kaggle_secrets import UserSecretsClient
from rich import print as rprint

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

login(token = HF_TOKEN)

api = HfApi()

REPO_ID = "Subhadip007/UERP_Dataset"

In [2]:
catalog_path = hf_hub_download(repo_id = REPO_ID, filename = "catalog_with_features_v1.parquet", repo_type = "dataset", token = HF_TOKEN)
catalog = pd.read_parquet(catalog_path)

structured_path = hf_hub_download(repo_id = REPO_ID, filename = "structured_features_v1.npy", repo_type = "dataset", token = HF_TOKEN)
structured_features = np.load(structured_path)

text_emb_path = hf_hub_download(repo_id = REPO_ID, filename = "overview_embeddings_v1.npy", repo_type = "dataset", token = HF_TOKEN)
text_embeddings = np.load(text_emb_path)

order_path = hf_hub_download(repo_id = REPO_ID, filename = "embedding_content_id_order.csv", repo_type = "dataset", token = HF_TOKEN)
embedding_order = pd.read_csv(order_path)

rprint("catalog:", catalog.shape)
rprint("structured_features:", structured_features.shape)
rprint("text_embeddings:", text_embeddings.shape)
rprint("embedding_order:", embedding_order.shape)

catalog_with_features_v1.parquet:   0%|          | 0.00/10.9M [00:00<?, ?B/s]

structured_features_v1.npy:   0%|          | 0.00/5.61M [00:00<?, ?B/s]

overview_embeddings_v1.npy:   0%|          | 0.00/120M [00:00<?, ?B/s]

embedding_content_id_order.csv: 0.00B [00:00, ?B/s]

catalog:
(38984, 53)

structured_features:
(38984, 36)

text_embeddings:
(38984, 768)

embedding_order:
(38984, 1)

In [3]:
match = (catalog['content_id'].reset_index(drop = True) == embedding_order['content_id'].reset_index(drop = True)).all()
rprint("Catalog and embedding order match:", match)

rprint("Row counts equal:", len(catalog) == structured_features.shape[0] == text_embeddings.shape[0] == len(embedding_order))

Catalog and embedding order match: True

Row counts equal: True

### **Stage 1: Popularity-Based Recommender**

In [4]:
def get_popular(catalog, content_type = None, is_anime = None, genre = None, top_n = None):
    df = catalog.copy()

    if content_type is not None:
        df = df[df['content_type'] == content_type]
    if is_anime is not None:
        df = df[df['is_anime'] == is_anime]
    if genre is not None:
        df = df[df['genres'].apply(lambda g: genre in g)]

    df = df.sort_values(['popularity_percentile', 'rating_normalized'], ascending = False)
    return df[['title', 'content_type', 'is_anime', 'genres', 'rating_normalized', 'popularity_percentile']].head(top_n)

# Test - overall top 10
rprint("--- Overall Top 10 ---")
rprint(get_popular(catalog, top_n = 10))
rprint()

# Test - top anime only
rprint("--- Top 10 Anime ---")
rprint(get_popular(catalog, is_anime = True, top_n = 10))
rprint()

# Test - top Horror genre
rprint("--- Top 10 Horror ---")
rprint(get_popular(catalog, genre = 'Horror', top_n = 10))

--- Overall Top 10 ---

title content_type  is_anime  \
0            The Shawshank Redemption        movie     False   
34093                 Attack on Titan    tv_series      True   
1                     The Dark Knight        movie     False   
2                           Inception        movie     False   
19857                    Breaking Bad    tv_series     False   
19858                 Game of Thrones    tv_series     False   
3                          Fight Club        movie     False   
4                        Interstellar        movie     False   
34094  Demon Slayer: Kimetsu no Yaiba    tv_series      True   
5                        Forrest Gump        movie     False   

                                                  genres  rating_normalized  \
0                                                [Drama]                9.3   
34093                  [Action, Drama, Fantasy, Mystery]                8.5   
1                                      [Crime, Thriller]                9.1   
2                 [Adventure, Science Fiction, Thriller]                8.8   
19857                           [Crime, Drama, Thriller]                9.5   
19858                                   [Drama, Fantasy]                9.2   
3                               [Crime, Drama, Thriller]                8.8   
4                    [Adventure, Drama, Science Fiction]                8.7   
34094  [Action, Adventure, Drama, Fantasy, Supernatural]                8.3   
5                                       [Drama, Romance]                8.8   

       popularity_percentile  
0                   1.000000  
34093               1.000000  
1                   0.999971  
2                   0.999941  
19857               0.999912  
19858               0.999883  
3                   0.999853  
4                   0.999824  
34094               0.999796  
5                   0.999795

--- Top 10 Anime ---

title content_type  is_anime  \
34093                 Attack on Titan    tv_series      True   
34094  Demon Slayer: Kimetsu no Yaiba    tv_series      True   
34095                  JUJUTSU KAISEN    tv_series      True   
34096                      Death Note    tv_series      True   
34097                My Hero Academia    tv_series      True   
34098          Hunter x Hunter (2011)    tv_series      True   
34099                   One-Punch Man    tv_series      True   
34100                       ONE PIECE    tv_series      True   
34101                     Tokyo Ghoul    tv_series      True   
34102        Attack on Titan Season 2    tv_series      True   

                                                  genres  rating_normalized  \
34093                  [Action, Drama, Fantasy, Mystery]                8.5   
34094  [Action, Adventure, Drama, Fantasy, Supernatural]                8.3   
34095                      [Action, Drama, Supernatural]                8.4   
34096   [Mystery, Psychological, Supernatural, Thriller]                8.4   
34097                        [Action, Adventure, Comedy]                7.7   
34098                       [Action, Adventure, Fantasy]                8.9   
34099    [Action, Comedy, Science Fiction, Supernatural]                8.3   
34100        [Action, Adventure, Comedy, Drama, Fantasy]                8.7   
34101  [Action, Drama, Horror, Mystery, Psychological...                7.6   
34102                  [Action, Drama, Fantasy, Mystery]                8.5   

       popularity_percentile  
34093               1.000000  
34094               0.999796  
34095               0.999591  
34096               0.999387  
34097               0.999182  
34098               0.998978  
34099               0.998773  
34100               0.998569  
34101               0.998364  
34102               0.998160

--- Top 10 Horror ---

title content_type  is_anime  \
18     The Silence of the Lambs        movie     False   
19859           Stranger Things    tv_series     False   
34101               Tokyo Ghoul    tv_series      True   
53                  The Shining        movie     False   
19860          The Walking Dead    tv_series     False   
74                        Alien        movie     False   
34111    The Promised Neverland    tv_series      True   
128                 I Am Legend        movie     False   
34113              Chainsaw Man    tv_series      True   
137             American Psycho        movie     False   

                                                  genres  rating_normalized  \
18                                [Crime, Drama, Horror]                8.6   
19859                           [Drama, Fantasy, Horror]                8.6   
34101  [Action, Drama, Horror, Mystery, Psychological...                7.6   
53                                       [Drama, Horror]                8.4   
19860                          [Drama, Horror, Thriller]                8.1   
74                             [Horror, Science Fiction]                8.4   
34111  [Drama, Fantasy, Horror, Mystery, Psychologica...                8.3   
128                     [Drama, Horror, Science Fiction]                7.2   
34113              [Action, Drama, Horror, Supernatural]                8.3   
137                               [Crime, Drama, Horror]                7.6   

       popularity_percentile  
18                  0.999413  
19859               0.999355  
34101               0.998364  
53                  0.998357  
19860               0.998181  
74                  0.997653  
34111               0.996320  
128                 0.995952  
34113               0.995911  
137                 0.995659

### **Stage 2: Hybrid Content-Based Recommender ("Similar to X")**

In [5]:
def get_similar(content_id, catalog, structured_features, text_embeddings, genre_weight = 0.5, text_weight = 0.5, top_n = 10):
    
    idx_matches = catalog.index[catalog['content_id'] == content_id]
    if len(idx_matches) == 0:
        return f"content_id '{content_id}' not found"
    idx = idx_matches[0]

    genre_vec = structured_features[idx, :32].reshape(1, -1)
    genre_sims = cosine_similarity(genre_vec, structured_features[:, :32])[0]

    text_vec = text_embeddings[idx].reshape(1, -1)
    text_sims = cosine_similarity(text_vec, text_embeddings)[0]

    hybrid_scores = (genre_weight * genre_sims) + (text_weight * text_sims)

    result_df = catalog.copy()
    result_df['similarity_score'] = hybrid_scores
    result_df = result_df[result_df['content_id'] != content_id]   
    result_df = result_df.sort_values('similarity_score', ascending = False)

    return result_df[['title', 'content_type', 'is_anime', 'genres', 'similarity_score']].head(top_n)

inception_id = catalog[catalog['title'] == 'Inception']['content_id'].values[0]
rprint(get_similar(inception_id, catalog, structured_features, text_embeddings, top_n = 10))

title content_type  is_anime  \
11698               Baida        movie     False   
10095          57 Seconds        movie     False   
27204       The Champions    tv_series     False   
245             Limitless        movie     False   
18907    Project 'Gemini'        movie     False   
4184             Stowaway        movie     False   
29461     The Lost Future     tv_movie     False   
3237      Mission to Mars        movie     False   
14749            The Wave        movie     False   
22979  Planet of the Apes    tv_series     False   

                                       genres  similarity_score  
11698  [Adventure, Science Fiction, Thriller]          0.814890  
10095  [Adventure, Science Fiction, Thriller]          0.759574  
27204  [Adventure, Science Fiction, Thriller]          0.756980  
245               [Science Fiction, Thriller]          0.734596  
18907  [Adventure, Science Fiction, Thriller]          0.726970  
4184   [Adventure, Science Fiction, Thriller]          0.725792  
29461  [Adventure, Science Fiction, Thriller]          0.725027  
3237   [Adventure, Science Fiction, Thriller]          0.709171  
14749             [Science Fiction, Thriller]          0.705658  
22979  [Adventure, Science Fiction, Thriller]          0.705617

In [6]:
def debug_similarity(content_id, target_titles, catalog, structured_features, text_embeddings):
    idx = catalog.index[catalog['content_id'] == content_id][0]
    genre_vec = structured_features[idx, :32].reshape(1, -1)
    text_vec = text_embeddings[idx].reshape(1, -1)

    for t in target_titles:
        matches = catalog[catalog['title'] == t]
        if matches.empty:
            continue
        t_idx = matches.index[0]
        g_sim = cosine_similarity(genre_vec, structured_features[t_idx, :32].reshape(1,-1))[0][0]
        tx_sim = cosine_similarity(text_vec, text_embeddings[t_idx].reshape(1,-1))[0][0]
        pop = catalog.loc[t_idx, 'popularity_percentile']
        rprint(f"{t:20s} | genre_sim = {g_sim:.3f} | text_sim = {tx_sim:.3f} | popularity = {pop:.4f}")

debug_similarity(inception_id, 
    ['Baida', '57 Seconds', 'The Champions', 'Total Recall', 'The Matrix', 'Memento', 'Source Code'],
    catalog, structured_features, text_embeddings)

Baida                | genre_sim = 1.000 | text_sim = 0.630 | popularity = 0.5785

57 Seconds           | genre_sim = 1.000 | text_sim = 0.519 | popularity = 0.6400

The Champions        | genre_sim = 1.000 | text_sim = 0.514 | popularity = 0.0545

Total Recall         | genre_sim = 0.667 | text_sim = 0.539 | popularity = 0.9818

The Matrix           | genre_sim = 0.408 | text_sim = 0.555 | popularity = 0.9997

Memento              | genre_sim = 0.000 | text_sim = 0.485 | popularity = 0.9989

Source Code          | genre_sim = 0.000 | text_sim = 0.480 | popularity = 0.9908

In [7]:
def get_similar(content_id, catalog, structured_features, text_embeddings, genre_weight = 0.5, text_weight = 0.5, min_popularity_percentile = 0.7,
                top_n = 10):
    
    idx_matches = catalog.index[catalog['content_id'] == content_id]
    if len(idx_matches) == 0:
        return f"content_id '{content_id}' not found"
    idx = idx_matches[0]

    genre_vec = structured_features[idx, :32].reshape(1, -1)
    genre_sims = cosine_similarity(genre_vec, structured_features[:, :32])[0]

    text_vec = text_embeddings[idx].reshape(1, -1)
    text_sims = cosine_similarity(text_vec, text_embeddings)[0]

    hybrid_scores = (genre_weight * genre_sims) + (text_weight * text_sims)

    result_df = catalog.copy()
    result_df['similarity_score'] = hybrid_scores
    result_df = result_df[result_df['content_id'] != content_id]

    result_df = result_df[result_df['popularity_percentile'] >= min_popularity_percentile]

    result_df = result_df.sort_values('similarity_score', ascending=False)
    return result_df[['title', 'content_type', 'is_anime', 'genres', 'rating_normalized', 'popularity_percentile', 'similarity_score']].head(top_n)

rprint(get_similar(inception_id, catalog, structured_features, text_embeddings, top_n = 10))

title content_type  is_anime  \
245          Limitless        movie     False   
4184          Stowaway        movie     False   
3237   Mission to Mars        movie     False   
7392           Seconds        movie     False   
1496         Companion        movie     False   
7661  Fantastic Voyage        movie     False   
5643      Subservience        movie     False   
62       Jurassic Park        movie     False   
209         Prometheus        movie     False   
6016            Cypher        movie     False   

                                      genres  rating_normalized  \
245              [Science Fiction, Thriller]                7.4   
4184  [Adventure, Science Fiction, Thriller]                5.7   
3237  [Adventure, Science Fiction, Thriller]                5.7   
7392             [Science Fiction, Thriller]                7.6   
1496             [Science Fiction, Thriller]                6.9   
7661            [Adventure, Science Fiction]                6.8   
5643             [Science Fiction, Thriller]                5.4   
62              [Adventure, Science Fiction]                8.2   
209             [Adventure, Science Fiction]                7.0   
6016    [Mystery, Science Fiction, Thriller]                6.7   

      popularity_percentile  similarity_score  
245                0.992139          0.734596  
4184               0.859590          0.725792  
3237               0.892265          0.709171  
7392               0.742440          0.692633  
1496               0.951398          0.692625  
7661               0.732643          0.663729  
5643               0.807585          0.656903  
62                 0.998035          0.654959  
209                0.993342          0.648314  
6016               0.793946          0.640357

In [8]:
def compare_weights(content_id, catalog, structured_features, text_embeddings, weight_options, min_pop = 0.7, top_n = 8):
    for gw in weight_options:
        tw = 1 - gw
        result = get_similar(content_id, catalog, structured_features, text_embeddings,
                             genre_weight = gw, text_weight = tw, min_popularity_percentile = min_pop, top_n = top_n)
        rprint(f"--- genre_weight = {gw}, text_weight = {tw:.1f} ---")
        rprint(list(result['title']))
        rprint()

compare_weights(inception_id, catalog, structured_features, text_embeddings, weight_options = [0.7, 0.5, 0.3, 0.15])

--- genre_weight = 0.7, text_weight = 0.3 ---

[
    'Stowaway',
    'Mission to Mars',
    'Limitless',
    'Seconds',
    'Companion',
    'Tengoku Daimakyo',
    'Fantastic Voyage',
    'Subservience'
]

--- genre_weight = 0.5, text_weight = 0.5 ---

[
    'Limitless',
    'Stowaway',
    'Mission to Mars',
    'Seconds',
    'Companion',
    'Fantastic Voyage',
    'Subservience',
    'Jurassic Park'
]

--- genre_weight = 0.3, text_weight = 0.7 ---

['Limitless', 'Seconds', 'Companion', 'Cypher', 'Primer', 'Total Recall', 'Stowaway', 'Brazil']

--- genre_weight = 0.15, text_weight = 0.8 ---

['Limitless', 'Paranoia', 'Parker', 'Cypher', 'Primer', 'Paprika', 'Total Recall', 'Seconds']

In [9]:
def get_similar(content_id, catalog, structured_features, text_embeddings, 
                genre_weight = 0.3, text_weight = 0.7,
                min_popularity_percentile = 0.7, top_n = 10):
    idx_matches = catalog.index[catalog['content_id'] == content_id]
    if len(idx_matches) == 0:
        return f"content_id '{content_id}' not found"
    idx = idx_matches[0]

    genre_vec = structured_features[idx, :32].reshape(1, -1)
    genre_sims = cosine_similarity(genre_vec, structured_features[:, :32])[0]

    text_vec = text_embeddings[idx].reshape(1, -1)
    text_sims = cosine_similarity(text_vec, text_embeddings)[0]

    hybrid_scores = (genre_weight * genre_sims) + (text_weight * text_sims)

    result_df = catalog.copy()
    result_df['similarity_score'] = hybrid_scores
    result_df = result_df[result_df['content_id'] != content_id]
    result_df = result_df[result_df['popularity_percentile'] >= min_popularity_percentile]
    result_df = result_df.sort_values('similarity_score', ascending = False)

    return result_df[['title', 'content_type', 'is_anime', 'genres', 'rating_normalized', 'popularity_percentile', 'similarity_score']].head(top_n)

In [10]:
fantasy_action_movie = catalog[(catalog['is_anime'] == False) & (catalog['genres'].apply(lambda g: 'Fantasy' in g and 'Action' in g))].sort_values('popularity_percentile', ascending = False).iloc[0]
rprint("Testing with:", fantasy_action_movie['title'])
test_id = fantasy_action_movie['content_id']

result = get_similar(test_id, catalog, structured_features, text_embeddings, genre_weight = 0.3, text_weight = 0.7, top_n = 15)
rprint(result[['title', 'is_anime', 'genres']])
rprint()
rprint("How many Anime came in top-15:", result['is_anime'].sum())

Testing with: Star Wars: Episode IV - A New Hope

title  is_anime  \
56         Star Wars: Episode VI - Return of the Jedi     False   
29     Star Wars: Episode V - The Empire Strikes Back     False   
108         Star Wars: Episode I - The Phantom Menace     False   
200           Star Wars: Episode VIII - The Last Jedi     False   
19877                                 The Mandalorian     False   
111      Star Wars: Episode III - Revenge of the Sith     False   
5412                                            Krull     False   
327     Star Wars: Episode IX - The Rise of Skywalker     False   
20061                                          Ahsoka     False   
35222               Jack-of-All-Trades, Party of None      True   
1278      Valerian and the City of a Thousand Planets     False   
20058                                     The Acolyte     False   
1132                   Godzilla: King of the Monsters     False   
32113                    Kenobi: A Star Wars Fan Film     False   
191          Pirates of the Caribbean: At World's End     False   

                                      genres  
56              [Action, Adventure, Fantasy]  
29     [Adventure, Fantasy, Science Fiction]  
108             [Action, Adventure, Fantasy]  
200             [Action, Adventure, Fantasy]  
19877           [Action, Adventure, Fantasy]  
111             [Action, Adventure, Fantasy]  
5412            [Action, Adventure, Fantasy]  
327             [Action, Adventure, Fantasy]  
20061           [Action, Adventure, Fantasy]  
35222           [Action, Adventure, Fantasy]  
1278            [Action, Adventure, Fantasy]  
20058           [Action, Adventure, Fantasy]  
1132            [Action, Adventure, Fantasy]  
32113           [Action, Adventure, Fantasy]  
191             [Action, Adventure, Fantasy]

How many Anime came in top-15: 1

In [11]:
import os

os.makedirs("/kaggle/working/processed", exist_ok = True)

In [12]:
%%writefile /kaggle/working/processed/recommender.py

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_popular(catalog, content_type = None, is_anime = None, genre = None, top_n = 10):
    df = catalog.copy()
    if content_type is not None:
        df = df[df['content_type'] == content_type]
    if is_anime is not None:
        df = df[df['is_anime'] == is_anime]
    if genre is not None:
        df = df[df['genres'].apply(lambda g: genre in g)]
    df = df.sort_values(['popularity_percentile', 'rating_normalized'], ascending = False)
    return df.head(top_n)

def get_similar(content_id, catalog, structured_features, text_embeddings,
                genre_weight = 0.3, text_weight = 0.7,
                min_popularity_percentile = 0.7, top_n = 10):
    idx_matches = catalog.index[catalog['content_id'] == content_id]
    if len(idx_matches) == 0:
        return None
    idx = idx_matches[0]

    genre_vec = structured_features[idx, :32].reshape(1, -1)
    genre_sims = cosine_similarity(genre_vec, structured_features[:, :32])[0]

    text_vec = text_embeddings[idx].reshape(1, -1)
    text_sims = cosine_similarity(text_vec, text_embeddings)[0]

    hybrid_scores = (genre_weight * genre_sims) + (text_weight * text_sims)

    result_df = catalog.copy()
    result_df['similarity_score'] = hybrid_scores
    result_df = result_df[result_df['content_id'] != content_id]
    result_df = result_df[result_df['popularity_percentile'] >= min_popularity_percentile]
    result_df = result_df.sort_values('similarity_score', ascending = False)

    return result_df.head(top_n)

Writing /kaggle/working/processed/recommender.py


In [13]:
import sys

sys.path.append('/kaggle/working/processed')
from recommender import get_popular, get_similar

test = get_similar(inception_id, catalog, structured_features, text_embeddings, top_n = 5)
rprint(test[['title']])

title
245   Limitless
7392    Seconds
1496  Companion
6016     Cypher
2227     Primer

In [14]:
room = catalog[catalog['title'] == 'The Room']

rprint(room[['title', 'rating_normalized', 'popularity_percentile']])

title  rating_normalized  popularity_percentile
2701  The Room                3.6               0.910715
6872  The Room                6.1               0.762239

In [15]:
def get_similar(content_id, catalog, structured_features, text_embeddings,
                 genre_weight=0.3, text_weight=0.7,
                 min_popularity_percentile=0.7, min_rating=6.0,
                 top_n=10):
    idx_matches = catalog.index[catalog['content_id'] == content_id]
    if len(idx_matches) == 0:
        return None
    idx = idx_matches[0]

    genre_vec = structured_features[idx, :32].reshape(1, -1)
    genre_sims = cosine_similarity(genre_vec, structured_features[:, :32])[0]
    text_vec = text_embeddings[idx].reshape(1, -1)
    text_sims = cosine_similarity(text_vec, text_embeddings)[0]
    hybrid_scores = (genre_weight * genre_sims) + (text_weight * text_sims)

    result_df = catalog.copy()
    result_df['similarity_score'] = hybrid_scores
    result_df = result_df[result_df['content_id'] != content_id]
    result_df = result_df[result_df['popularity_percentile'] >= min_popularity_percentile]
    result_df = result_df[result_df['rating_normalized'] >= min_rating]
    result_df = result_df.sort_values('similarity_score', ascending=False)
    return result_df.head(top_n)

# Re-test Shawshank
shawshank_id = catalog[catalog['content_id']=='imdb_tt0111161']['content_id'].values[0]
rprint(get_similar(shawshank_id, catalog, structured_features, text_embeddings, top_n=10)[['title','rating_normalized','similarity_score']])

title  rating_normalized  similarity_score
7123   The United States of Leland                6.9          0.735787
5656                  The Woodsman                7.1          0.721768
5359                         Boy A                7.5          0.715535
2781                   Half Nelson                7.1          0.715181
1428                     25th Hour                7.6          0.712678
7565                     Innocence                8.0          0.707326
6000                        A Hero                7.5          0.706603
4478                        Palmer                7.3          0.705680
20775                      Rectify                8.3          0.703663
35010  Showa Genroku Rakugo Shinju                8.4          0.703184

In [16]:
debug_similarity(
    catalog[catalog['title']=='The Shawshank Redemption']['content_id'].values[0],
    ['The Green Mile', 'The United States of Leland', 'A Hero'],
    catalog, structured_features, text_embeddings
)

The Green Mile       | genre_sim = 0.577 | text_sim = 0.604 | popularity = 0.9991

The United States of Leland | genre_sim = 1.000 | text_sim = 0.623 | popularity = 0.7530

A Hero               | genre_sim = 1.000 | text_sim = 0.581 | popularity = 0.7946